# Streaming updates

This notebook sends a surface in pieces. It first sends the component tree with an empty list, then appends restaurants one at a time, then changes an existing item and adds a footer. Each `send` call is validated and applied in order, so the page fills in while the cell runs.

In [4]:
from renderer_setup import new_surface

surface = new_surface("streaming-updates")
surface.send(
    [
        {
            "version": "1.0",
            "updateDataModel": {
                "surfaceId": "streaming-updates",
                "value": {"restaurants": []},
            },
        },
        {
            "version": "1.0",
            "updateComponents": {
                "surfaceId": "streaming-updates",
                "components": [
                    {
                        "id": "root",
                        "component": "Column",
                        "children": ["heading", "restaurant_list"],
                    },
                    {"id": "heading", "component": "Text", "text": "## Restaurants"},
                    {
                        "id": "restaurant_list",
                        "component": "Column",
                        "children": {"path": "/restaurants", "componentId": "restaurant_card"},
                    },
                    {
                        "id": "restaurant_card",
                        "component": "Column",
                        "children": ["card_title", "card_subtitle"],
                    },
                    {"id": "card_title", "component": "Text", "text": {"@path": "title"}},
                    {
                        "id": "card_subtitle",
                        "component": "Text",
                        "text": {"@path": "subtitle"},
                        "variant": "caption",
                    },
                ],
            },
        },
    ]
)
surface

This cell appends one restaurant every second. Each `updateDataModel` targets the next index in `/restaurants`, and the list template adds a card for it.

In [5]:
import asyncio

restaurants = [
    {"title": "The Golden Fork", "subtitle": "Fine dining and spirits"},
    {"title": "Ocean's Bounty", "subtitle": "Fresh daily seafood"},
    {"title": "Pizzeria Roma", "subtitle": "Wood-fired pizza"},
]

for index, restaurant in enumerate(restaurants):
    surface.send(
        {
            "version": "1.0",
            "updateDataModel": {
                "surfaceId": "streaming-updates",
                "path": f"/restaurants/{index}",
                "value": restaurant,
            },
        }
    )
    await asyncio.sleep(1)

This cell changes the subtitle of the first restaurant in place and adds a footer. Sending `updateComponents` again with the same ids replaces those components and the root's child list.

In [6]:
surface.send(
    [
        {
            "version": "1.0",
            "updateDataModel": {
                "surfaceId": "streaming-updates",
                "path": "/restaurants/0/subtitle",
                "value": "Fine dining, now open late",
            },
        },
        {
            "version": "1.0",
            "updateComponents": {
                "surfaceId": "streaming-updates",
                "components": [
                    {
                        "id": "root",
                        "component": "Column",
                        "children": ["heading", "restaurant_list", "footer"],
                    },
                    {
                        "id": "footer",
                        "component": "Text",
                        "text": "Updated in place by a later cell",
                        "variant": "caption",
                    },
                ],
            },
        },
    ]
)